# Model B Eğitimi — Sürücü Eylemleri / Yolcu / Nesne

**13 class:** arka_koltuk_1, arka_koltuk_2, arkaya_bakma, bilgisayar,
emniyet_kemeri_ihlali, esneme, etrafa_bakinma, **kemer_takili** *(JSON'a yazılmaz)*,
on_koltuk, sigara_icme, su_icme, teknocan, telefonla_konusma

Bu notebook'u çalıştırmadan önce:
- `Runtime > Change runtime type > T4 GPU` seç
- Her hücreyi **sırayla** çalıştır

In [ ]:
# ── HÜCRE 1: GPU Kontrolü ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('GPU bulunamadı! Runtime > Change runtime type > T4 GPU seç')
print(result.stdout)
print('✓ GPU hazır')

Sun Jun 28 06:26:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── HÜCRE 2: Drive Bağla ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive bağlandı')

Mounted at /content/drive
✓ Drive bağlandı


In [ ]:
# ── HÜCRE 3: ZIP Dosyasını Bul ─────────────────────────────────────────────
# Model B ZIP'inin adını buraya yaz (Drive'da nasıl görünüyorsa)
# Örnek: 'surucu_eylemleri.v1i.yolov8.zip'
MODEL_B_ZIP_NAME = ''   # <── BURAYA ZIP ADINI YAZ, boş bırakırsan otomatik arar

import os, zipfile, glob

if MODEL_B_ZIP_NAME:
    search_paths = glob.glob(f'/content/drive/MyDrive/**/{MODEL_B_ZIP_NAME}', recursive=True)
    search_paths += glob.glob(f'/content/drive/MyDrive/{MODEL_B_ZIP_NAME}')
else:
    # Otomatik ara — Drive'daki tüm yolov8 zip'lerini bul
    search_paths = glob.glob('/content/drive/MyDrive/**/*yolov8*.zip', recursive=True)
    search_paths += glob.glob('/content/drive/MyDrive/*yolov8*.zip')
    # Model A zip'ini listeden çıkar
    search_paths = [p for p in search_paths if 'Rumii' not in p and 'model_a' not in p.lower()]

if not search_paths:
    print('Drive içeriği:')
    for f in sorted(os.listdir('/content/drive/MyDrive/')):
        if f.endswith('.zip'):
            print(f'  {f}')
    raise FileNotFoundError(
        'Model B ZIP bulunamadı!\n'
        'MODEL_B_ZIP_NAME değişkenine ZIP dosyasının tam adını yaz.'
    )

if len(search_paths) > 1:
    print('Birden fazla ZIP bulundu, ilki kullanılacak:')
    for p in search_paths:
        print(f'  {p}')

ZIP_PATH = search_paths[0]
print(f'✓ ZIP bulundu: {ZIP_PATH}')

DATASET_DIR = '/content/dataset_b'
os.makedirs(DATASET_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

print(f'✓ ZIP çıkartıldı: {DATASET_DIR}')
for item in sorted(os.listdir(DATASET_DIR)):
    print(f'  {item}')

Birden fazla ZIP bulundu, ilki kullanılacak:
  /content/drive/MyDrive/ENson.v2i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/hedef.v1i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/hedef.v1i.yolov8-obb (1).zip
  /content/drive/MyDrive/Veri Setleri İKA/water filled barrier.v9i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/ika.v2i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/Ika-Parkur-Objects.v3i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/lazer.v3i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/Target.v2i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/ika_tabelalar.v2i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/Etap Sembolleri.v3i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/tabela_tespit.v8i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/Bariyer.v4i.yolov8-obb.zip
  /content/drive/MyDrive/Veri Setleri İKA/birlesmisdataset.v1i.yolov8-obb.zip
  /content/drive/My

In [ ]:
# ── HÜCRE 4: data.yaml Yolunu Bul ve Düzelt ───────────────────────────────
import os, glob, yaml

yaml_files = glob.glob(f'{DATASET_DIR}/**/data.yaml', recursive=True)
yaml_files += glob.glob(f'{DATASET_DIR}/data.yaml')

if not yaml_files:
    raise FileNotFoundError(f'data.yaml bulunamadı! {DATASET_DIR} içini kontrol et')

DATA_YAML    = yaml_files[0]
DATASET_ROOT = os.path.dirname(DATA_YAML)
print(f'✓ data.yaml: {DATA_YAML}')
print(f'✓ Dataset kök: {DATASET_ROOT}')

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('\nOrijinal data.yaml:')
print(yaml.dump(data_cfg, default_flow_style=False))

# Yolları Colab'a göre güncelle
for split in ['train', 'val', 'valid', 'test']:
    if split in data_cfg:
        old_path = data_cfg[split]
        if not old_path.startswith('/'):
            new_path = os.path.join(DATASET_ROOT, old_path)
        else:
            rel = old_path.lstrip('/')
            if rel.startswith(('train', 'val', 'test')):
                new_path = os.path.join(DATASET_ROOT, rel)
            else:
                new_path = os.path.join(DATASET_ROOT, os.path.basename(old_path))
        data_cfg[split] = new_path

FIXED_YAML = '/content/data_b.yaml'
with open(FIXED_YAML, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, allow_unicode=True)

print('\nGüncellenmiş data.yaml:')
with open(FIXED_YAML) as f:
    print(f.read())

for split in ['train', 'val', 'valid']:
    if split in data_cfg:
        p = data_cfg[split]
        print(f'  {split}: {"✓ VAR" if os.path.exists(p) else "✗ YOK"} → {p}')

✓ data.yaml: /content/dataset_b/data.yaml
✓ Dataset kök: /content/dataset_b

Orijinal data.yaml:
names:
  0: arka_koltuk_1
  1: arka_koltuk_2
  2: arkaya_bakma
  3: bilgisayar
  4: emniyet_kemeri_ihlali
  5: esneme
  6: etrafa_bakinma
  7: kemer_takili
  8: on_koltuk
  9: sigara_icme
  10: su_icme
  11: teknocan
  12: telefonla_konusma
test: test/images
train: train/images
val: valid/images


Güncellenmiş data.yaml:
names:
  0: arka_koltuk_1
  1: arka_koltuk_2
  2: arkaya_bakma
  3: bilgisayar
  4: emniyet_kemeri_ihlali
  5: esneme
  6: etrafa_bakinma
  7: kemer_takili
  8: on_koltuk
  9: sigara_icme
  10: su_icme
  11: teknocan
  12: telefonla_konusma
test: /content/dataset_b/test/images
train: /content/dataset_b/train/images
val: /content/dataset_b/valid/images

  train: ✓ VAR → /content/dataset_b/train/images
  val: ✓ VAR → /content/dataset_b/valid/images


In [ ]:
# ── HÜCRE 5: Class Sırası Kontrolü ────────────────────────────────────────
# kemer_takili=7 dahil — inference'ta filtrelenecek ama eğitimde gerekli
EXPECTED_CLASSES_B = [
    'arka_koltuk_1',           # 0
    'arka_koltuk_2',           # 1
    'arkaya_bakma',            # 2
    'bilgisayar',              # 3
    'emniyet_kemeri_ihlali',   # 4
    'esneme',                  # 5
    'etrafa_bakinma',          # 6
    'kemer_takili',            # 7  ← JSON'a yazılmaz, kontrast sağlar
    'on_koltuk',               # 8
    'sigara_icme',             # 9
    'su_icme',                 # 10
    'teknocan',                # 11
    'telefonla_konusma',       # 12
]

with open(FIXED_YAML) as f:
    cfg = yaml.safe_load(f)

names = cfg.get('names', {})
actual = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)

print(f'Class sayısı: {len(actual)} (beklenen: {len(EXPECTED_CLASSES_B)})')
print('\nClass sırası:')
for i, name in enumerate(actual):
    expected = EXPECTED_CLASSES_B[i] if i < len(EXPECTED_CLASSES_B) else '???'
    note = ' ← JSON\'a yazılmaz' if name == 'kemer_takili' else ''
    match = '✓' if name == expected else f'✗ (beklenen: {expected})'
    print(f'  {i}: {name:<30} {match}{note}')

if actual == EXPECTED_CLASSES_B:
    print('\n✓ Class sırası doğru!')
else:
    print('\n⚠️  Class sırası farklı — configs/model_b_config.yaml güncellenmeli')
    print('   Gerçek sıra:', actual)

Class sayısı: 13 (beklenen: 13)

Class sırası:
  0: arka_koltuk_1                  ✓
  1: arka_koltuk_2                  ✓
  2: arkaya_bakma                   ✓
  3: bilgisayar                     ✓
  4: emniyet_kemeri_ihlali          ✓
  5: esneme                         ✓
  6: etrafa_bakinma                 ✓
  7: kemer_takili                   ✓ ← JSON'a yazılmaz
  8: on_koltuk                      ✓
  9: sigara_icme                    ✓
  10: su_icme                        ✓
  11: teknocan                       ✓
  12: telefonla_konusma              ✓

✓ Class sırası doğru!


In [ ]:
# ── HÜCRE 6: Ultralytics Kur ───────────────────────────────────────────────
!pip install -q ultralytics
import ultralytics
print(f'✓ Ultralytics {ultralytics.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Ultralytics 8.4.80


In [ ]:
# ── HÜCRE 7: Eğitim Konfigürasyonu ────────────────────────────────────────
import torch

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name   = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name} ({gpu_mem_gb:.0f} GB)')

if gpu_mem_gb >= 35:    # A100
    BATCH = 32
elif gpu_mem_gb >= 14:  # T4 / L4
    BATCH = 16
else:
    BATCH = 8

TRAIN_CFG = {
    'model':          'yolov8s.pt',   # small: kabin içi detay için yeterli
    'data':           FIXED_YAML,
    'epochs':         50,
    'imgsz':          640,
    'batch':          BATCH,
    'patience':       20,
    'project':        '/content/runs/model_b',
    'name':           'train',
    'exist_ok':       True,
    'device':         0,
    'workers':        2,
    'cache':          True,
    'save':           True,
    'save_period':    10,
    'plots':          True,
    'optimizer':      'AdamW',
    'lr0':            0.001,
    'lrf':            0.01,
    'weight_decay':   0.0005,
    'warmup_epochs':  3,
    'cos_lr':         True,
    'label_smoothing': 0.1,
    # Augmentation — kabin içi: ışık çok değişken
    'hsv_h':     0.015,
    'hsv_s':     0.8,    # daha agresif — tünel/gece/gündüz
    'hsv_v':     0.6,    # daha agresif — parlaklık değişimi
    'degrees':   3.0,    # kabin içi kamera az döner
    'translate': 0.1,
    'scale':     0.5,
    'flipud':    0.0,
    'fliplr':    0.0,    # Sürücü koltuk yönü flip ile karışmamalı
    'mosaic':    1.0,
    'mixup':     0.1,
}

print(f'\nEğitim konfigürasyonu:')
print(f'  Model  : {TRAIN_CFG["model"]}')
print(f'  Epoch  : {TRAIN_CFG["epochs"]}')
print(f'  Batch  : {TRAIN_CFG["batch"]}')
print(f'  imgsz  : {TRAIN_CFG["imgsz"]}')
print(f'  Çıktı  : {TRAIN_CFG["project"]}/{TRAIN_CFG["name"]}')

GPU: Tesla T4 (16 GB)

Eğitim konfigürasyonu:
  Model  : yolov8s.pt
  Epoch  : 50
  Batch  : 16
  imgsz  : 640
  Çıktı  : /content/runs/model_b/train


In [ ]:
# ── HÜCRE 8: Eğitimi Başlat ────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(TRAIN_CFG.pop('model'))
results = model.train(**TRAIN_CFG)

print('\n✓ Eğitim tamamlandı!')
print(f'  mAP50    : {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')
print(f'  mAP50-95 : {results.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_b.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.8, hsv_v=0.6, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=N

In [ ]:
# ── HÜCRE 9: Ağırlıkları Drive'a Kaydet ───────────────────────────────────
# Colab session kapanınca /content/ SİLİNİR — MUTLAKA çalıştır!
import shutil, os
from pathlib import Path

SAVE_DIR = '/content/drive/MyDrive/teknofest_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

best_pt = Path('/content/runs/model_b/train/weights/best.pt')
last_pt = Path('/content/runs/model_b/train/weights/last.pt')

if best_pt.exists():
    dst = f'{SAVE_DIR}/model_b_best.pt'
    shutil.copy(best_pt, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'✓ best.pt kaydedildi: {dst} ({size_mb:.1f} MB)')
else:
    print('✗ best.pt bulunamadı!')

if last_pt.exists():
    shutil.copy(last_pt, f'{SAVE_DIR}/model_b_last.pt')
    print(f'✓ last.pt kaydedildi')

plots_src = Path('/content/runs/model_b/train')
plots_dst = Path(f'{SAVE_DIR}/model_b_plots')
plots_dst.mkdir(exist_ok=True)
for png in plots_src.glob('*.png'):
    shutil.copy(png, plots_dst / png.name)
print(f'✓ Grafikler: {plots_dst}')

print(f'\n✓ Drive\'/MyDrive/teknofest_weights/ içinde:')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  {f}')

✗ best.pt bulunamadı!
✓ Grafikler: /content/drive/MyDrive/teknofest_weights/model_b_plots

✓ Drive'/MyDrive/teknofest_weights/ içinde:
  model_b_plots


In [ ]:
# ── HÜCRE 10: Sonuçları Görüntüle ─────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path
import yaml

run_dir = Path('/content/runs/model_b/train')

for fname, title in [
    ('results.png',          'Eğitim Grafikleri'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png',         'PR Eğrisi'),
]:
    p = run_dir / fname
    if p.exists():
        print(f'── {title} ──')
        display(Image(str(p)))

model = YOLO('/content/runs/model_b/train/weights/best.pt')
metrics = model.val(data=FIXED_YAML, device=0, verbose=False)
with open(FIXED_YAML) as f:
    names_cfg = yaml.safe_load(f).get('names', {})
names_list = [names_cfg[i] for i in sorted(names_cfg)] if isinstance(names_cfg, dict) else list(names_cfg)

print('\n── Class Bazında mAP50 ──')
if hasattr(metrics.box, 'ap_class_index'):
    for i, idx in enumerate(metrics.box.ap_class_index):
        name  = names_list[idx] if idx < len(names_list) else str(idx)
        ap    = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
        note  = ' ← JSON\'a yazılmaz' if name == 'kemer_takili' else ''
        bar   = '█' * int(ap * 30)
        print(f'  {name:<30} {ap:.3f}  {bar}{note}')

NameError: name 'YOLO' is not defined